#### Desnormalizando datos de nuestra base de datos SQL del delivery para juntar información relevante en un solo objeto JSON/TEXTO Optimizado para: Búsqueda semántica, contexto completo, embeddings

In [28]:
#motor semántico + gobernanza + aprendizaje incremental
import os
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from dotenv import load_dotenv


# Cargar variables de entorno
load_dotenv()

# === DATOS DE CONEXIÓN ===
DB_USER = os.getenv("DB_USER")
DB_PASS = quote_plus(os.getenv("DB_PASS", "")) # se codifica aquí
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# === ENGINE ===
engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    pool_pre_ping=True
)

# === PRUEBA DE CONEXIÓN ===
with engine.connect() as conn:
    print("✅ Conexión exitosa a MySQL")

# === CARGAR TABLA ===
negocios = pd.read_sql("""
SELECT
  id,
  name,
  description,
  horario_apertura,
  horario_cierre,
  diasOperacion,
  direccion,
  created_at,
  updated_at,
  descripcion_detallada
FROM categories
""", engine)

negocios.head()

✅ Conexión exitosa a MySQL


,id,name,description,horario_apertura,horario_cierre,diasOperacion,direccion,created_at,updated_at,descripcion_detallada
0,31,TACOS LOS LICENCIADOS,Tacos de carne asada,0 days 12:02:14,0 days 16:24:56,Diario,"Progreso, segunda sección",2026-01-23 08:48:23,2025-09-23 21:54:51,Negocio de comida que ofrece tacos de asada en...
1,33,02-PIZZA CENTRAL,"Pizzas, papas y hamburguesas",0 days 13:21:26,0 days 19:43:15,Lunes a Viernes,"Calle 96, Ciudad 50",2026-01-23 14:08:54,2025-09-23 21:57:17,None
2,34,DESAYUNOS GAHORY,Desayunos en general,0 days 09:03:39,0 days 17:33:48,Miércoles a Domingo,"Calle 65, Ciudad 43",2026-01-23 14:06:24,2025-09-23 21:58:13,None
3,35,02-EXPRESSO CENTRAL,"Cafetería, crepas y más",0 days 11:04:55,0 days 21:53:39,Lunes a Domingo,"Calle 15, Ciudad 33",2026-01-23 14:08:39,2025-09-23 21:59:43,None
4,36,"HAMBURGUESAS ""EL GÜERO""","Hamburguesas, papas, hot-dogs",0 days 16:24:01,0 days 14:21:40,Jueves a Domingo,"Calle 60, Ciudad 50",2026-01-23 08:48:28,2025-09-23 22:00:19,Negocio de comida rápida que ofrece hamburgues...


In [29]:
productos = pd.read_sql("""
SELECT
  id,
  name,
  description,
  price,
  image1,
  image2,
  image3,
  id_category
FROM products
""", engine)

productos.head()

,id,name,description,price,image1,image2,image3,id_category
0,38,ENCHILADAS(DESAYUNOS GAHORY),CAMPIRANO:VERDES:POLLO / HUEVO,70.0,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,34
1,39,ENCHILADAS NORTEÑA(DESAYUNOS GAHORY),ROJAS C/PECHUGA ASADA ENCEBOLLADA,85.0,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,34
2,40,ENCHILADAS SUIZO(DESAYUNOS GAHORY),EN SALSA CREMOSITA GRATINADAS EN QUESO MANCHEGO,95.0,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,34
3,41,ENCHILADAS DELICIA(DESAYUNOS GAHORY),DIVORCIADAS,85.0,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,34
4,42,ENCHILADAS PAISA(DESAYUNOS GAHORY),ENCHILADAS EN SALSA DE FRIJOL,85.0,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,https://firebasestorage.googleapis.com/v0/b/te...,34


##### La parte de desnormalizar (preparado el JSON para el RAG): 




In [30]:
import re
import pandas as pd

# ============================================
# Prepara el JSON base desde SQL (LIMPIEZA DE DATOS)
# ============================================

docs = []

def clean_time(t):
    if pd.isna(t): #Convierte valores vacíos a None para que el JSON generado quede limpio, sin basura.
        return None

    if isinstance(t, pd.Timedelta): #Sirve para convertir tiempos SQL tipo 00:00:00 a formato estándar HH:MM.
        total_seconds = int(t.total_seconds())
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        return f"{hours:02d}:{minutes:02d}" #Esta linea sirve para que evite segundos, ideal para IA, POWERBI Y BUSQUEDAS SEMANTICAS

    if isinstance(t, str): #Corta segundos sin validar de más,por ejemlpo: de 08:00:00 queda 08:00
        return t[:5]

    if hasattr(t, "strftime"): #si el objeto se comporta como una hora/fecha, elimina. Este patrón soporta varios tipos sin preguntarle su clase exacta
        return t.strftime("%H:%M") #En estas dos lineas sirven para soportar objetos datetime.time entre otros

    return None


def clean_text(t): #Limpieza general de texto, sirve para que eliminar ruido en los datos, evitar campos vacíos en embeddings y mejorar la precisión en RAG
    if pd.isna(t) or t is None or str(t).strip() == "":
        return None
    return str(t).strip() # Esta linea ayuda a quitar espacios antes y después y normaliza textos para IA, por ejemplo: "  Taquería familiar  " a "Taquería familiar"


#Limpieza EXCLUSIVA para el nombre del negocio (clave para búsquedas y embeddings)
#EJEMPLO:
def clean_nombre(t):
    if pd.isna(t) or t is None or str(t).strip() == "":
        return None

    t = str(t).strip()

    # Quita prefijos tipo: 01-, 02 -, 123 -
    t = re.sub(r'^\d+\s*-\s*', '', t)

    return t


for _, n in negocios.iterrows():
    prods = productos[productos.id_category == n.id]

    descripcion_corta = clean_text(n.description)
    descripcion_detallada = clean_text(n.descripcion_detallada)

    # Fallback inteligente para descripción
    if not descripcion_detallada and descripcion_corta:
        descripcion_detallada = f"Negocio de comida que ofrece {descripcion_corta.lower()}."

    # Aquí aplicas la limpieza correcta
    nombre = clean_nombre(n["name"])


    docs.append({
        # ─────────────────────────────
        # IDENTIDAD
        # ─────────────────────────────
        "document_id": f"negocio_{n.id}",
        "tipo": "negocio",
        "nombre": nombre,

        # ─────────────────────────────
        # DESCRIPCIONES
        # ─────────────────────────────
        "descripcion_corta": descripcion_corta,
        "descripcion_detallada": descripcion_detallada,

        # ─────────────────────────────
        # UBICACIÓN
        # ─────────────────────────────
        "direccion": clean_text(n.direccion),

        # ─────────────────────────────
        # HORARIOS
        # ─────────────────────────────
        "horarios": {
            "dias": clean_text(n.diasOperacion),
            "apertura": clean_time(n.horario_apertura),
            "cierre": clean_time(n.horario_cierre)
        },

        # ─────────────────────────────
        # PRODUCTOS
        # ─────────────────────────────
        "productos": (
            prods[["name", "description", "price"]]
            .rename(columns={
                "name": "nombre",
                "description": "descripcion",
                "price": "precio"
            })
            .map(clean_text)
            .to_dict("records")
        )
    })

In [31]:
#PARA EXPORTAR EL JSON, EL PRIMERO,
#CON LOS DATOS ORIGINALES
import json

with open("datos.json", "w", encoding="utf-8") as f:
    json.dump(docs, f, ensure_ascii=False, indent=2)

print("✅ Dataset RAG generado")

✅ Dataset RAG generado


In [ ]:
###############LEER IMPORTANTE#################
# SISTEMA diseñado para aprender de forma robusta.

# Cada corrida analiza toda la plataforma (cross-negocio)

# Guarda todo en labels_pendientes.json con evidencia

# Si activas AUTO_PROMOTE_LABEL=True, promueve solo si:

# aparece en muchos negocios

# tiene muchas menciones

# y no es un término basura

#T odo queda guardado en label_registry.json y se usa en corridas futuras
#######################
#¿QUE ES?
# ✅ un motor de conocimiento incremental
# ✅ con memoria persistente
# ✅ con reglas + evidencia
# ✅ listo para RAG / embeddings / búsqueda semántica

#######################################

# El fin de todo este bloque es convertir texto sucio de negocios 
# y productos en un sistema de clasificación que aprende solo, 
# recuerda lo aprendido y solo evoluciona cuando hay evidencia 
# real a nivel plataforma.

###########################
# Es un sistema de clasificación semántica con aprendizaje heurístico global 
# y gobernanza automática, usado como preprocesador para RAG
###########################
# Genera docs_rag.json (negocio + producto) robusto y auto-actualizable:
# - Reglas base (LABEL_RULES)
# - Conceptos canónicos (CANONICAL_CONCEPTS) -> tags/booleans
# - Aprendizaje GLOBAL cross-negocio: descubre nuevos términos/phrases
# - Registro persistente de labels/tags: label_registry.json
# - Pendientes con evidencia: labels_pendientes.json
#
# Ideal para: Qwen3 + embeddings + Chroma/FAISS + n8n

import json
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from datetime import datetime

# =========================
# Config
# =========================
registry = {
    "labels_oficiales": [],
    "auto_promoted": []
}
#######
def term_key(term: str) -> str:
    """
    Clave canónica para gobernanza semántica.
    NO limpia acentos.
    NO elimina caracteres.
    SOLO controla mayúsculas y espacios.
    """
    if not term:
        return ""
    return term.strip().lower()

#####


INPUT_DATOS = "datos.json"
OUTPUT_DOCS_RAG = "docs_rag.json"

REGISTRY_PATH = "label_registry.json" 
#label_registry.json no se recalcula desde cero
# Contiene:
# -labels_oficiales: etiquetas confiables para UI, filtros, dashboards
# -auto_promoted: historial de etiquetas que el sistema promovió solo
# Sin esto, cada corrida sería “amnesia total”.
# Con esto, el sistema madura con el tiempo.


PENDING_PATH = "labels_pendientes.json"
# Sirve para:
# -auditar
# -ajustar umbrales
# -aprobar manualmente
# -entender qué está aprendiendo el sistema

PRODUCT_GOV_PATH = "product_term_governance.json"


TOP_TAGS_PER_BIZ = 24
TOP_CANDIDATES_PER_BIZ = 12

MAX_NGRAMS = 2  # 1 = unigramas, 2 = bigramas, 3 = trigramas 
#(los n-grams covierten el texto en unidades analizables)
# Y tmbien los n-grams representan texto como datos

# =========================
# REGISTRO DE GOBERNANZA GLOBAL
# =========================


# =========================
# Gobernanza semántica productos (GLOBAL)
# =========================
PRODUCT_TERM_GOVERNANCE = {}

# Umbrales de “aprendizaje global”
# (ajustar según tamaño de la plataforma)
MIN_BUSINESSES_FOR_TAG = 2        # para aceptar tags globales
MIN_BUSINESSES_FOR_LABEL = 5      # para proponer como label candidata (categoría)
AUTO_PROMOTE_LABEL = True         # si True, promueve labels cuando pasa umbral
AUTO_PROMOTE_MIN_BUSINESSES = 8   # negocios distintos
AUTO_PROMOTE_MIN_TOTAL_MENTIONS = 20  # menciones totales (tokens/ngrams)
AUTO_PROMOTE_MIN_LEN = 4          # mínimo largo para auto-promoción

# =========================
# Stopwords / ruido
# =========================
STOPWORDS = {
    "de","la","el","y","o","con","sin","para","por","en","a","al","del","los","las",
    "un","una","unos","unas","que","se","su","sus","tipo","ofrece","negocio","comida",
    "incluye","pieza","piezas","orden","ml","lt","l","litro","litros","media","medio",
    "grande","familiar","especial","clasica","clasico","clásico","clasica","sencilla",
    "sencillo","combo","paquete","promocion","promoción","envio","envío","gratis",
    "desde","hasta","precio","precios","aprox","aproximado","a","elegir"
}

GENERIC_MENU_WORDS = {
    "rico","rica","delicioso","deliciosa","sabroso","sabrosa","caliente","crujiente",
    "suave","casero","casera","artesanal","premium","favorito","favorita",
    "fresco","fresca","natural","hecho","hecha","especialidad","tradicional"
}

UNITS = {"kg","kilo","kilos","gr","g","ml","lt","l","litro","litros","pz","pza","pzas","pieza","piezas"}

# =========================
# Normalización
# =========================
def normalize(text: str) -> str:
    if not text:
        return ""
    text = text.lower().strip()
    # text = unicodedata.normalize("NFD", text)
    # text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    #text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str):
    text = normalize(text)
    toks = []
    for t in text.split():
        if not t or t in STOPWORDS:
            continue
        if len(t) < 3:
            continue
        if t.isdigit():
            continue
        if t in UNITS:
            continue
        if t in GENERIC_MENU_WORDS:
            continue
        toks.append(t)
    return toks

def make_ngrams(tokens, n=2):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def is_bad_term(term: str) -> bool:
    if not term:
        return True

    parts = term.split()

    # dígitos SOLO si es token único
    if len(parts) == 1 and any(ch.isdigit() for ch in term):
        return True

    if len(parts) == 1:
        if len(term) < 3:
            return True
        if term in STOPWORDS or term in GENERIC_MENU_WORDS or term in UNITS:
            return True

    if term in STOPWORDS or term in GENERIC_MENU_WORDS or term in UNITS:
        return True

    return False

################################333

def dias_a_numeros(dias_str: str):
    """
    Convierte texto libre de días a lista [0-6]
    lunes = 0, domingo = 6
    Función ROBUSTA para texto real de negocios.
    """
    if not dias_str:
        return []

    text = normalize(dias_str)

    day_map = {
        "lunes": 0, "lun": 0,
        "martes": 1, "mar": 1,
        "miercoles": 2, "miércoles": 2, "mie": 2,
        "jueves": 3, "jue": 3,
        "viernes": 4, "vie": 4,
        "sabado": 5, "sábado": 5, "sab": 5,
        "domingo": 6, "dom": 6
    }

    # =========================
    # 1 Casos globales
    # =========================
    if any(x in text for x in ["todos los dias", "todos los días", "diario", "diaria"]):
        return list(range(7))

    if "fin de semana" in text:
        return [5, 6]

    # =========================
    # 2 Rangos explícitos (lunes a viernes, mar-dom, etc)
    # =========================
    dias = set()

    for start_name, start_idx in day_map.items():
        for end_name, end_idx in day_map.items():
            if f"{start_name} a {end_name}" in text or f"{start_name}-{end_name}" in text:
                if start_idx <= end_idx:
                    dias.update(range(start_idx, end_idx + 1))
                else:
                    # rango cruzando semana (ej: viernes a lunes)
                    dias.update(range(start_idx, 7))
                    dias.update(range(0, end_idx + 1))

    # =========================
    # 3 Listas simples (lunes, martes y jueves)
    # =========================
    for name, idx in day_map.items():
        if re.search(rf"\b{name}\b", text):
            dias.add(idx)

    # =========================
    # 4 Exclusiones simples (excepto martes)
    # =========================
    if "excepto" in text and dias:
        excluidos = set()
        for name, idx in day_map.items():
            if re.search(rf"\b{name}\b", text):
                excluidos.add(idx)
        dias -= excluidos

        return sorted(dias)
####################################
from datetime import datetime, time
def parse_hora(hora_str: str):
    """
    Convierte texto de hora a datetime.time
    Acepta:
    - HH
    - HH:MM
    - H
    - H:MM
    - HH:MM hrs
    """
    if not hora_str:
        return None

    s = normalize(hora_str)
    s = re.sub(r"[^\d:]", "", s)

    if not s:
        return None

    try:
        if ":" in s:
            h, m = s.split(":", 1)
            return time(int(h), int(m))
        else:
            return time(int(s), 0)
    except Exception:
        return None

    ######################


def is_abierto_ahora(dias_str, apertura_str, cierre_str, now=None):
    """
    Determina si el negocio está abierto en este momento.

    Retorna:
    - True  → abierto
    - False → cerrado
    - None  → información insuficiente
    """

    if not dias_str or not apertura_str or not cierre_str:
        return None  # ❗ no inventar

    if not now:
        now = datetime.now()

    dias_abiertos = dias_a_numeros(dias_str)
    if not dias_abiertos:
        return None

    apertura = parse_hora(apertura_str)
    cierre = parse_hora(cierre_str)
    if not apertura or not cierre:
        return None

    now_time = now.time()
    hoy = now.weekday()  # lunes = 0

    # =========================
    # CASO 1 — horario normal
    # ej: 09:00 - 18:00
    # =========================
    if apertura < cierre:
        return (
            hoy in dias_abiertos
            and apertura <= now_time <= cierre
        )

    # =========================
    # CASO 2 — cruza medianoche
    # ej: 18:00 - 02:00
    # =========================
    else:
        ayer = (hoy - 1) % 7

        return (
            (hoy in dias_abiertos and now_time >= apertura)
            or
            (ayer in dias_abiertos and now_time <= cierre)
        )
    

################################
def abre_el_dia(dias_str: str, target_day: int) -> bool | None:
    """
    Verifica si el negocio abre un día específico.
    target_day: lunes=0 ... domingo=6
    """
    if not dias_str:
        return None

    dias = dias_a_numeros(dias_str)
    if not dias:
        return None

    return target_day in dias


def abre_hoy(dias_str: str, now=None):
    if not now:
        now = datetime.now()
    return abre_el_dia(dias_str, now.weekday())


def abre_manana(dias_str: str, now=None):
    if not now:
        now = datetime.now()
    manana = (now.weekday() + 1) % 7
    return abre_el_dia(dias_str, manana)


def abre_fin_de_semana(dias_str: str):
    dias = dias_a_numeros(dias_str)
    if not dias:
        return None
    return any(d in dias for d in (5, 6))


def abre_de_noche(apertura_str: str, threshold="18:00"):
    apertura = parse_hora(apertura_str)
    limite = parse_hora(threshold)

    if not apertura or not limite:
        return None

    return apertura >= limite


def cierra_tarde(cierre_str: str, threshold="22:00"):
    cierre = parse_hora(cierre_str)
    limite = parse_hora(threshold)

    if not cierre or not limite:
        return None

    # Si cierra después de medianoche, es tarde
    if cierre < limite:
        return True

    return cierre >= limite

####################################
# =========================
# Labels reglas (base)
# =========================
LABEL_RULES = {
    "tacos": ["taco", "tacos", "taqueria", "taquería", "pastor", "suadero", "bistec", "arrachera", "longaniza", "chistorra"],
    "pizza": ["pizza", "pepperoni", "hawai", "hawaiana", "margarita", "tokio", "suprema"],
    "hamburguesas": ["hamburguesa", "burger", "hotdog", "hot-dog", "hot dog", "papas", "salchipulpos", "salchicha", "boneless", "alitas"],
    "desayunos": ["desayuno", "chilaquiles", "huevos", "hotcakes", "omelette", "enchiladas", "molletes", "cafe de olla", "tamales", "atole"],
    "cafeteria": ["cafe", "café", "capuchino", "latte", "moka", "frappe", "chai", "matcha", "chocolate caliente", "crepa", "crepas"],
    "mariscos": ["mariscos", "coctel", "cóctel", "camaron", "camarón", "mojarra", "ceviche", "aguachile"],
    "antojitos": ["quesadilla", "quesadillas", "sopes", "sope", "gorditas", "tlacoyo", "tostada", "flautas", "gringas", "huarache", "chalupas"],
    "postres": ["pastel", "rebanada", "galletas", "waffles", "crepa", "nutella", "cajeta", "bombones", "granola", "flan", "gelatina", "churros", "helado"],
    "bebidas": ["refresco", "coca", "cocacola", "boing", "agua", "limonada", "jugo", "te", "té", "malteada", "horchata", "jamaica", "tamarindo"],
    "alcohol": ["vinateria", "vinatería", "cerveza", "six", "laton", "latón", "tequila", "mezcal", "whisky", "red label", "pulque", "michelada", "chelada"],
}

# =========================
# Conceptos canónicos (alias -> tag fuerte)
# =========================
CANONICAL_CONCEPTS = {
    "comida corrida": [
        "comida corrida", "cocina economica", "cocina económica",
        "fonda", "comedor", "menu del dia", "menú del día",
        "guisados", "comida casera", "corrida"
    ],
    "tacos de canasta": ["tacos de canasta", "tacos sudados", "tacos al vapor"],
    "barbacoa": ["barbacoa", "consome de barbacoa", "consomé de barbacoa"],
    "birria": ["birria", "consome", "consomé"],
    "carnitas": ["carnitas"],
    "cochinita pibil": ["cochinita", "pibil", "cochinita pibil"],
}

def apply_canonical_tags(full_text: str):
    norm = normalize(full_text)
    tags = set()

    for canon, variants in CANONICAL_CONCEPTS.items():
        hits = 0
        for v in variants:
            if normalize(v) in norm:
                hits += 1
        if hits >= 1:
            tags.add(canon)

    return tags

def collapse_semantic_terms(terms):
    """
    Reduce ruido semántico:
    - prioriza n-grams largos
    - elimina tokens contenidos
    """
    terms = list(dict.fromkeys(terms))
    terms_sorted = sorted(terms, key=lambda x: -len(x.split()))

    collapsed = []

    for t in terms_sorted:
        is_subterm = False
        for other in terms_sorted:
            if t != other and f" {t} " in f" {other} ":
                is_subterm = True
                break
        if not is_subterm:
            collapsed.append(t)

    return sorted(collapsed, key=lambda x: (-len(x.split()), x))

####Esta parte, lo que hace es una decisión de arquitectura semántica.####
#El objetivo es construir un resumen semántico del negocio que represente
#qué vende realmente, no de qué está hecho.
#El principio es : "Un término NO es un producto vendible si su comportamiento estadístico
        #indica que es componente, atributo o modificador, no una entidad comercial"
def is_product_like_term(term: str) -> bool:
    """
    Decide si un término representa un producto vendible real.
    NO decide gobernanza, solo naturaleza semántica.
    """
    term_norm = normalize(term)

    info = PRODUCT_TERM_GOVERNANCE.get(term_norm)

    # 🔹 Término nuevo → NO descartes
    if not info:
        return True

    # 🔹 Ingredientes / atributos
    if (
        info["businesses"] >= 10 and
        info["total_mentions"] >= 30 and
        info["is_unigram"] and
        info["length"] <= 10
    ):
        return False

    # 🔹 Demasiado genérico
    if (
        info["businesses"] >= 15 and
        info["total_mentions"] >= 50 and
        info["length"] <= 8
    ):
        return False

    return True

###############################
def is_governed_product_term(term: str) -> bool:
    """
    Decide si un término puede entrar a docs_rag.json
    (conocimiento estable, no experimental).
    """

    term_norm = normalize(term)

    # 1-Labels oficiales a SIEMPRE válidos
    if term_norm in [normalize(x) for x in registry.get("labels_oficiales", [])]:
        return True

    info = PRODUCT_TERM_GOVERNANCE.get(term_norm)
    if not info:
        return False

    # 2-Frases compuestas con evidencia fuerte (bigrams / trigrams)
    if (
        not info["is_unigram"] and
        info["businesses"] >= MIN_BUSINESSES_FOR_LABEL and
        info["total_mentions"] >= AUTO_PROMOTE_MIN_TOTAL_MENTIONS
    ):
        return True

    return False
####################################
# IMPORTANTE:
# productos_representativos SOLO contiene términos gobernados.
# NO es un extractor de texto, es un resumen semántico estable para RAG.
def build_products_semantic_summary(biz: dict, max_items=6):
    terms = []

    for p in biz.get("productos", []) or []:
        name = (p or {}).get("nombre", "")
        desc = (p or {}).get("descripcion", "")
        text = f"{name} {desc}"

        toks = tokenize(text)

        for t in toks:
            if (
                not is_bad_term(t)
                and is_product_like_term(t)
                and is_governed_product_term(t)
            ):
                terms.append(t)

        if MAX_NGRAMS >= 2:
            bgs = make_ngrams(toks, 2)
            for bg in bgs:
                if (
                    not is_bad_term(bg)
                    and is_product_like_term(bg)
                    and is_governed_product_term(bg)
                ):
                    terms.append(bg)

    if not terms:
        return ""

    cnt = Counter(terms)

    ranked = sorted(
        cnt.items(),
        key=lambda x: (
            -x[1],
            -len(x[0].split()),  # 🔥 prioriza bigramas
            x[0]
        )
    )

    top = [t for t, _ in ranked]
    top = collapse_semantic_terms(top)

    return ", ".join(top[:max_items])
# =========================
# Construcción texto
# =========================
def build_full_text_business(biz: dict) -> str:
    nombre = str(biz.get("nombre", "") or "")
    dc = str(biz.get("descripcion_corta", "") or "")
    dd = str(biz.get("descripcion_detallada", "") or "")
    prod_text_parts = []
    for p in biz.get("productos", []) or []:
        prod_text_parts.append(str((p or {}).get("nombre", "") or ""))
        prod_text_parts.append(str((p or {}).get("descripcion", "") or ""))
    return " ".join([nombre, dc, dd, " ".join(prod_text_parts)])

def build_full_text_product(biz: dict, p: dict) -> str:
    return " ".join([
        str(biz.get("nombre", "") or ""),
        str(biz.get("descripcion_corta", "") or ""),
        str((p or {}).get("nombre", "") or ""),
        str((p or {}).get("descripcion", "") or "")
    ])

# =========================
# Aprendizaje global (cross-negocio)
# =========================

def learn_global_terms(data):
    """
    El fin: detectar patrones a nivel plataforma, no casos aislados.
    Aquí es donde se hace la magia para que se autoalimente de conceptos
    de negocios nuevos.

    Retorna:
    - term_counts: conteo total (tokens y ngrams)
    - term_businesses: set de negocios donde aparece cada término
    - examples: ejemplos (nombre de negocio)
    """

    term_counts = Counter()
    term_businesses = defaultdict(set)
    examples = defaultdict(list)

    for biz in data:
        negocio_id = biz.get("document_id")
        if not negocio_id:
            continue

        full = build_full_text_business(biz)
        toks = tokenize(full)

        bgs = []
        tgs = []

        # ======================
        # TOKENS
        # ======================
        uniq_tokens = set(toks)
        for t in uniq_tokens:
            if is_bad_term(t):
                continue
            k = term_key(t)
            term_businesses[k].add(negocio_id)

        for t in toks:
            if is_bad_term(t):
                continue
            k = term_key(t)
            term_counts[k] += 1

        # ======================
        # N-GRAMS
        # ======================
        if MAX_NGRAMS >= 2:
            bgs = make_ngrams(toks, 2)

            for bg in bgs:
                if is_bad_term(bg):
                    continue
                k = term_key(bg)
                term_counts[k] += 1

            for bg in set(bgs):
                if is_bad_term(bg):
                    continue
                k = term_key(bg)
                term_businesses[k].add(negocio_id)

        if MAX_NGRAMS >= 3:
            for tg in tgs:
                if is_bad_term(tg):
                    continue
                k = term_key(tg)
                term_counts[k] += 1

            for tg in set(tgs):
                if is_bad_term(tg):
                    continue
                k = term_key(tg)
                term_businesses[k].add(negocio_id)

        # ======================
        # EJEMPLOS (tokens)
        # ======================
        biz_name = normalize(biz.get("nombre", "") or "")

        for t in list(uniq_tokens)[:20]:
            k = term_key(t)
            if len(examples[k]) < 3:
                examples[k].append(biz_name)

        # ======================
        # EJEMPLOS (bigramas)
        # ======================
        if MAX_NGRAMS >= 2:
            for bg in set(bgs):
                if is_bad_term(bg):
                    continue
                if len(examples[bg]) < 3:
                    examples[bg].append(biz_name)

        # ======================
        # EJEMPLOS (trigramas)
        # ======================
        if MAX_NGRAMS >= 3:
            for tg in set(tgs):
                if is_bad_term(tg):
                    continue
                if len(examples[tg]) < 3:
                    examples[tg].append(biz_name)

                    assert set(term_counts.keys()) == set(term_businesses.keys()) #Solo en desarrollo


    return term_counts, term_businesses, examples
    


def build_product_term_governance(term_counts, term_businesses):
   
    # Construye evidencia global para decidir si un término
    # representa un producto vendible real o solo un componente.
   
    governance = {}

    for term_keyed, total_mentions in term_counts.items():
        biz_count = len(term_businesses.get(term_keyed, []))

        governance[term_keyed] = {
            "total_mentions": total_mentions,
            "businesses": biz_count,
            "is_unigram": len(term_keyed.split()) == 1,
            "length": len(term_keyed),
        }

    return governance

def merge_product_governance(old, new):
    """
    Fusión acumulativa:
    - suma menciones
    - une negocios
    - conserva flags estructurales
    """
    merged = dict(old)

    for term, info in new.items():
        if term not in merged:
            merged[term] = info
            continue

        merged[term]["total_mentions"] += info["total_mentions"]
        merged[term]["businesses"] = info["businesses"]


    return merged


# =========================
# Registro persistente (labels oficiales y promociones)
# =========================
def load_registry():
    if not Path(REGISTRY_PATH).exists():
        return {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "labels_oficiales": sorted(list(LABEL_RULES.keys())),
    "tags_oficiales": [],
    "auto_promoted": [],
    "notes": {
        "labels_oficiales": "Categorías fuertes para UI y filtros",
        "tags_oficiales": "Vocabulario semántico estable (no categorías)"
        }
    }


    return json.loads(Path(REGISTRY_PATH).read_text(encoding="utf-8"))

def save_registry(registry):
    Path(REGISTRY_PATH).write_text(json.dumps(registry, ensure_ascii=False, indent=2), encoding="utf-8")

def load_product_term_governance():
    if not Path(PRODUCT_GOV_PATH).exists():
        return {}
    return json.loads(Path(PRODUCT_GOV_PATH).read_text(encoding="utf-8"))

def save_product_term_governance(governance):
    Path(PRODUCT_GOV_PATH).write_text(
        json.dumps(governance, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

# =========================
# Extracción local (por negocio) con apoyo global
# =========================
def extract_tags_local(full_text: str, global_terms=None):
    toks = tokenize(full_text)
    cnt = Counter(toks)

    if MAX_NGRAMS >= 2:
        cnt.update(make_ngrams(toks, 2))

    items = []
    for w, c in cnt.items():
        if is_bad_term(w):
            continue

        if global_terms:
            w_norm = normalize(w)
            if w_norm not in global_terms:
                continue

        items.append((w, c))

    items.sort(key=lambda x: (-x[1], x[0]))
    return [w for w, _ in items[:TOP_TAGS_PER_BIZ]]

def extract_candidates_from_global(term_counts, term_businesses, registry):
    """
    Produce candidatas "de plataforma" (robustas), no solo del negocio.
    """
    pending = []

    labels_oficiales = set([normalize(x) for x in registry.get("labels_oficiales", [])])
    base_labels = set([normalize(x) for x in LABEL_RULES.keys()])

    for term, total_count in term_counts.items():
        if is_bad_term(term):
            continue

        biz_count = len(term_businesses.get(term, set()))
        if biz_count < MIN_BUSINESSES_FOR_TAG:
            continue

        # Decide si es candidata a label (categoría) por evidencia fuerte
        is_label_candidate = biz_count >= MIN_BUSINESSES_FOR_LABEL

        # Evita promover cosas que ya son label base/oficial
        term_norm = normalize(term)
        already_label = (term_norm in labels_oficiales) or (term_norm in base_labels)

        pending.append({
            "term": term,
            "term_norm": term_norm,
            "total_mentions": int(total_count),
            "businesses": int(biz_count),
            "kind": (
            "label"
            if (is_label_candidate and not already_label and biz_count >= AUTO_PROMOTE_MIN_BUSINESSES)
            else "tag"
            ),
            "already_label": bool(already_label),
        })

    pending.sort(key=lambda x: (-x["businesses"], -x["total_mentions"], x["term_norm"]))
    return pending


# Un término no se pierde, no todo se vuelve categoría, la semántica crece sin ensuciar la UI
# labels_oficiales vuelve a ser sagrado
#EJEMPLO: tacos	- label| sushi - label | birria - tag | pastor - tag | artesanal - tag


def maybe_auto_promote(pending, registry):
    if not AUTO_PROMOTE_LABEL:
        return registry

    labels_oficiales = set(
        normalize(x) for x in registry.get("labels_oficiales", [])
    )
    tags_oficiales = set(
        normalize(x) for x in registry.get("tags_oficiales", [])
    )

    for item in pending:
        if item["already_label"]:
            continue
        if item["businesses"] < AUTO_PROMOTE_MIN_BUSINESSES:
            continue
        if item["total_mentions"] < AUTO_PROMOTE_MIN_TOTAL_MENTIONS:
            continue
        if len(item["term_norm"]) < AUTO_PROMOTE_MIN_LEN:
            continue

        # 🔷 PROMOCIÓN A LABEL (categoría)
        if (
            item["kind"] == "label"
            and not is_bad_term(item["term_norm"])
            and not is_product_like_term(item["term_norm"])  # 🔥 CLAVE
        ):
            if item["term_norm"] not in labels_oficiales:
                registry.setdefault("labels_oficiales", []).append(item["term_norm"])
                registry.setdefault("auto_promoted", []).append({
                    "type": "label",
                    "value": item["term_norm"],
                    "businesses": item["businesses"],
                    "total_mentions": item["total_mentions"],
                    "promoted_at": datetime.utcnow().isoformat() + "Z"
                })
                labels_oficiales.add(item["term_norm"])

        # 🔹 PROMOCIÓN A TAG (descriptor semántico)
        elif item["kind"] == "tag":
            if item["term_norm"] not in tags_oficiales:
                registry.setdefault("tags_oficiales", []).append(item["term_norm"])
                registry.setdefault("auto_promoted", []).append({
                    "type": "tag",
                    "value": item["term_norm"],
                    "businesses": item["businesses"],
                    "total_mentions": item["total_mentions"],
                    "promoted_at": datetime.utcnow().isoformat() + "Z"
                })
                tags_oficiales.add(item["term_norm"])

    # Normalización final
    registry["labels_oficiales"] = sorted(labels_oficiales)
    registry["tags_oficiales"] = sorted(tags_oficiales)

    return registry


# =========================
# Clasificación por reglas + tags/canónicos + labels oficiales
# =========================
def generate_labels_tags_for_business(biz: dict, registry):
    full_text = build_full_text_business(biz)
    norm_full = normalize(full_text)

    # =========================
    # LABELS (clasificación fuerte)
    # =========================
    labels = set()

    # 1 labels por reglas base
    for label, kws in LABEL_RULES.items():
        for kw in kws:
            if normalize(kw) in norm_full:
                labels.add(label)
                break


    # =========================
    # TAGS (detalle semántico)
    # =========================
    # 3️⃣ tags canónicos (conceptos fuertes, diseñados)
    canonical_tags = apply_canonical_tags(full_text)

    # 4️⃣ tags locales gobernados (aprendidos)
    local_tags = set(
        extract_tags_local(
            full_text,
            global_terms=global_terms
        )
    )

    governed_tags = {
        t for t in local_tags
        if normalize(t) in registry.get("tags_oficiales", [])
    }

    # 5️⃣ 🔹 VISTA COMBINADA (OPCIONAL)
    tags = canonical_tags | governed_tags

    # =========================
    # Reglas especiales
    # =========================
    if "comida corrida" in tags:
        labels.add("comida_casera")

    return sorted(labels), sorted(canonical_tags), sorted(governed_tags)

def join_str(values):
    if not values:
        return ""
    return "|".join([normalize(str(v)) for v in values])
########################



# =========================
# MAIN
# =========================
def main():
    data = json.loads(Path(INPUT_DATOS).read_text(encoding="utf-8"))

    if len(data) < 10:
        print("Dataset pequeño: aprendizaje global limitado")

    # 1) Aprendizaje global (cross-negocio)
    term_counts, term_businesses, examples = learn_global_terms(data)

 

    # =========================
    # Gobernanza productos (GLOBAL)
    # =========================
    global PRODUCT_TERM_GOVERNANCE

    # 1) cargar gobernanza previa
    prev_governance = load_product_term_governance()

    # 2) construir gobernanza de esta corrida
    current_governance = build_product_term_governance(
        term_counts,
        term_businesses
    )

    # 3) merge acumulativo
    PRODUCT_TERM_GOVERNANCE = merge_product_governance(
        prev_governance,
        current_governance
    )

    # 4) persistir
    save_product_term_governance(PRODUCT_TERM_GOVERNANCE)

    ###################################

    # Términos válidos a nivel plataforma (gobernanza global)
       
    global global_terms


    global_terms = {
    term
    for term, info in PRODUCT_TERM_GOVERNANCE.items()
    if (
        info["businesses"] >= MIN_BUSINESSES_FOR_TAG
        and info["total_mentions"] >= 2
        )
    }


    # 2) Registro persistente
    global registry
    registry = load_registry()

    # 3) Pendientes globales y auto-promoción (opcional)
    pending = extract_candidates_from_global(term_counts, term_businesses, registry)
    registry = maybe_auto_promote(pending, registry)
    save_registry(registry)

    # 4) Guardar pendientes con ejemplos
    # añade ejemplos (máximo 3)
    for it in pending[:500]:
        ex = examples.get(it["term"], [])[:3]
        it["examples_businesses"] = ex

    Path(PENDING_PATH).write_text(
        json.dumps({
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "min_businesses_for_tag": MIN_BUSINESSES_FOR_TAG,
            "min_businesses_for_label": MIN_BUSINESSES_FOR_LABEL,
            "auto_promote_label": AUTO_PROMOTE_LABEL,
            "pending": pending[:500],
        }, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    # 5) Generación docs negocio + producto
    docs = []

    for biz in data:
        negocio_id = biz.get("document_id") or biz.get("negocio_id")
        if not negocio_id:
            continue

        labels, canonical_tags, governed_tags = generate_labels_tags_for_business(biz, registry)
        tags = set(canonical_tags) | set(governed_tags)  # vista combinada
        
        labels_oficiales = [
            l for l in labels
            if normalize(l) in registry.get("labels_oficiales", [])
        ]

        is_comida_corrida = ("comida corrida" in tags)
        is_tacos = ("tacos" in labels)
        is_pizza = ("pizza" in labels)
        is_hamburguesas = ("hamburguesas" in labels)

        # CATEGORÍA PRINCIPAL (ancla semántica) 
        # Es solo un refuerzo semántico explícito para embeddings
        # =========================
        categoria_principal = None

        if is_tacos:
            categoria_principal = "tacos"
        elif is_pizza:
            categoria_principal = "pizza"
        elif is_hamburguesas:
            categoria_principal = "hamburguesas"
        elif is_comida_corrida:
            categoria_principal = "comida corrida"
        elif labels:
            # fallback inteligente si no entra en las anteriores
            categoria_principal = labels[0]



        # ========= Doc negocio =========
        business_text = build_full_text_business(biz)

        #Búsqueda humana simple (refuerzo semántico para embeddings)

        busqueda_simple = []

        def push(term):
            if term and term not in busqueda_simple:
                busqueda_simple.append(term)

        # 1️⃣ Categoría principal
        push(categoria_principal)

        # 2️⃣ Labels (mezcladas, pero estables)
        for lbl in labels:
            push(lbl)

        # 3️⃣ Tags (detalle fino)
        for tg in tags:
            push(tg)

        busqueda_simple = collapse_semantic_terms(busqueda_simple)
        busqueda_simple_str = ", ".join(busqueda_simple)

        # Horarios explícitos para IA
        horarios = biz.get("horarios") or {}
        dias = horarios.get("dias", "")
        dias_norm = normalize(dias)
        apertura = horarios.get("apertura", "")
        cierre = horarios.get("cierre", "")
        abierto_ahora = is_abierto_ahora(dias, apertura, cierre)

        now = datetime.now()

            # =========================
            # FEATURES TEMPORALES DERIVADAS
            # =========================

        abre_hoy_val = abre_hoy(dias, now)
        abre_manana_val = abre_manana(dias, now)
        abre_finde_val = abre_fin_de_semana(dias)
        abre_noche_val = abre_de_noche(apertura)
        cierra_tarde_val = cierra_tarde(cierre)


        text_lines = [
            "tipo_doc: negocio",
            f"negocio_id: {negocio_id}",
            f"nombre: {biz.get('nombre','')}",
        ]

        # Refuerzo semántico SOLO si hay certeza
        if abre_hoy_val is True:
            text_lines.append("abre_hoy: si")

        if abre_manana_val is True:
            text_lines.append("abre_manana: si")

        if abre_finde_val is True:
            text_lines.append("abre_fin_de_semana: si")

        if abre_noche_val is True:
            text_lines.append("abre_de_noche: si")

        if cierra_tarde_val is True:
            text_lines.append("cierra_tarde: si")

        #Refuerzo semántico SOLO si hay certeza
        if abierto_ahora is True:
            text_lines.append("estado_actual: abierto ahora")
        elif abierto_ahora is False:
            text_lines.append("estado_actual: cerrado ahora")


        # categoria principal SOLO si existe
        if categoria_principal:
            text_lines.append(f"categoria_principal: {categoria_principal}")

    
        text_lines.extend([
            f"descripcion: {biz.get('descripcion_corta','')} {biz.get('descripcion_detallada','')}",
            f"direccion: {biz.get('direccion','')}",

            f"horarios: {dias} {apertura} {cierre}",
            f"dias_abierto: {dias}",
            f"dias_abierto_norm: {dias_norm}",
            f"abre_a: {apertura}",
            f"cierra_a: {cierre}",

            f"busqueda_simple: {busqueda_simple_str}",

            f"labels_oficiales: {join_str(labels_oficiales) if labels_oficiales else ''}",

            f"tags_canonic: {join_str(canonical_tags) if canonical_tags else ''}",
            f"tags_gobernados: {join_str(governed_tags) if governed_tags else ''}",
        ])

        productos_resumen = build_products_semantic_summary(biz)

        if productos_resumen:
            text_lines.append(f"productos_representativos: {productos_resumen}")

        text_negocio = "\n".join(text_lines)



        docs.append({
            "id": f"{negocio_id}",
            "text": text_negocio,
            "metadata": {
                "tipo_doc": "negocio",
                "negocio_id": negocio_id,
                "nombre": biz.get("nombre"),
                "direccion": biz.get("direccion"),
                "dias": (biz.get("horarios") or {}).get("dias"),
                "apertura": (biz.get("horarios") or {}).get("apertura"),
                "cierre": (biz.get("horarios") or {}).get("cierre"),



                "labels": sorted(list(labels)),


                #  separación semántica clara
                "tags_canonic": sorted(list(canonical_tags)),

                "tags_gobernados": sorted(list(governed_tags)),


                #  vista combinada SOLO para búsqueda rápida
                "tags": sorted(list(tags)),


                #  versiones string (útiles para filtros / debug)
                "labels_str": join_str(labels),
                "tags_canonic_str": join_str(canonical_tags),
                "tags_gobernados_str": join_str(governed_tags),
                "tags_str": join_str(sorted(list(tags))),



                
                

                
                
                "is_comida_corrida": bool(is_comida_corrida),
                "is_tacos": bool(is_tacos),
                "is_pizza": bool(is_pizza),
                "is_hamburguesas": bool(is_hamburguesas),

                # info para debug/observabilidad
                "labels_oficiales_count": len(registry.get("labels_oficiales", [])),
                "tags_oficiales_count": len(registry.get("tags_oficiales", [])),
                "is_abierto_ahora": abierto_ahora,
                "horario_valido": abierto_ahora is not None,

                "abre_hoy": abre_hoy_val,
                "abre_manana": abre_manana_val,
                "abre_fin_de_semana": abre_finde_val,
                "abre_de_noche": abre_noche_val,
                "cierra_tarde": cierra_tarde_val,



            }
        })

        # ========= Docs PRODUCTO =========
        for idx, p in enumerate(biz.get("productos", []) or []):
            prod_name = (p or {}).get("nombre")
            prod_desc = (p or {}).get("descripcion")
            prod_price = (p or {}).get("precio")
            if not (prod_name or prod_desc):
                continue

            product_id = f"{negocio_id}::producto_{idx+1}"
            product_text_raw = build_full_text_product(biz, p)

            text_producto = "\n".join([
                "tipo_doc: producto",
                f"negocio_id: {negocio_id}",
                f"nombre_negocio: {biz.get('nombre','')}",
                f"producto_id: {product_id}",
                f"producto: {prod_name or ''}",
                f"descripcion_producto: {prod_desc or ''}",
                f"precio: {prod_price or ''}",
                f"labels_oficiales: {join_str(labels_oficiales) if labels_oficiales else ''}",
                f"tags_canonic: {join_str(canonical_tags) if canonical_tags else ''}",
                f"tags_gobernados: {join_str(governed_tags) if governed_tags else ''}",
                f"texto: {normalize(product_text_raw)}"
            ])

            docs.append({
                "id": product_id,
                "text": text_producto,
                "metadata": {
                    "tipo_doc": "producto",
                    "negocio_id": negocio_id,
                    "nombre_negocio": biz.get("nombre"),
                    "producto_id": product_id,
                    "producto_nombre": prod_name,
                    "producto_precio": float(prod_price) if prod_price else None,

                    "labels": sorted(list(labels)),


                    "tags_canonic": sorted(list(canonical_tags)),

                    "tags_gobernados": sorted(list(governed_tags)),

                    "tags": sorted(list(tags)),

                    "labels_str": join_str(labels),
                    "tags_canonic_str": join_str(canonical_tags),
                    "tags_gobernados_str": join_str(governed_tags),
                    "tags_str": join_str(sorted(list(tags))),


                    "is_comida_corrida": bool(is_comida_corrida),
                    "is_tacos": bool(is_tacos),
                    "is_pizza": bool(is_pizza),
                    "is_hamburguesas": bool(is_hamburguesas),
                }
            })

    Path(OUTPUT_DOCS_RAG).write_text(
        json.dumps(docs, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f"OK -> generado {OUTPUT_DOCS_RAG} con {len(docs)} docs (negocio + productos).")
    print(f"OK -> generado {PENDING_PATH} (pendientes globales).")
    print(f"OK -> generado {REGISTRY_PATH} (registro persistente).")

if __name__ == "__main__":
    main()

    ##### Texto listo para embeddings (Convierte cada negocio en un texto listo para embeddings.) 
    #Es aprendizaje automatico de docs_rag.json, en pocas palabras es ML

C:\Users\ulise\AppData\Local\Temp\ipykernel_20792\1972307421.py:1053: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",


OK -> generado docs_rag.json con 634 docs (negocio + productos).
OK -> generado labels_pendientes.json (pendientes globales).
OK -> generado label_registry.json (registro persistente).


#### La ultima parte, corrrer el programa que genera los embeddings del documento final docs_rag.json

In [34]:
%run generate_embeddings.py


c:\Users\ulise\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo all-MiniLM-L6-v2...
Documentos cargados: 634


Generando embeddings: 100%|██████████| 634/634 [01:14<00:00,  8.46it/s]


Embeddings guardados en docs_with_embeddings.json
